# Construction of a Reproducible 300-Sample MMMU-Pro Evaluation Subset

## Purpose

This notebook constructs a fixed 300-item evaluation subset from the official
**MMMU-Pro, Standard (10 options), test split**.

The sampling design enforces:

1. **Subject balance:** exactly 10 evaluation items from each of the 30 MMMU-Pro subjects.
2. **Target difficulty balance:** nominally 3 Easy, 4 Medium, and 3 Hard items per subject.
3. **Few-shot-first reconciliation:** if Easy or Hard becomes insufficient after excluding the supplied few-shot candidate pool, the notebook first reclaims the minimum number of missing Easy/Hard examples from that pool.
4. **Medium compensation only as a fallback:** any Easy/Hard deficit that still remains after same-difficulty reclamation—because the pinned MMMU-Pro source itself does not contain enough items—is transferred to Medium.
5. **Final few-shot separation:** every reclaimed item is permanently removed from the final few-shot pool and written to an explicit removal file.

Thus, every subject contributes exactly 10 items and the final evaluation set contains exactly

\[
30 \times 10 = 300
\]

items.

## Priority rule

For each subject and for each edge difficulty (`Easy`, `Hard`):

1. Try to satisfy the nominal quota of 3 using rows **outside** the original few-shot pool.
2. If fewer than 3 remain, reclaim exactly the missing number from the original 113-item few-shot pool **at the same subject and same difficulty**, up to what exists in the pinned source.
3. If the source itself still cannot provide 3 items, transfer only the irreducible remainder to `Medium`.

Therefore the policy is:

```text
missing Easy  -> reclaim Easy from 113 first -> residual deficit -> Medium
missing Hard  -> reclaim Hard from 113 first -> residual deficit -> Medium
```

This ordering preserves the intended 3/4/3 distribution whenever the source benchmark makes it possible, while still guaranteeing a 10-item subject total when a source-level difficulty shortage exists.

### Source-level limitations in the pinned MMMU-Pro split

The complete pinned source contains only:

- **1 Easy** item for `Electronics`;
- **1 Hard** item for `Literature`.

Consequently, no few-shot reclamation can restore those strata to 3. Their unavoidable residual deficits are compensated by Medium.

## Reproducibility principles

- MMMU-Pro is loaded from a pinned immutable revision.
- The master seed is fixed at **42**.
- Candidate rows are canonically sorted by benchmark `id`.
- Sampling is without replacement.
- The supplied 113-item few-shot candidate file is authenticated by SHA-256.
- Reclamation uses a separate deterministic PCG64 stream.
- Evaluation ordering uses another independent deterministic stream.
- Every reclaimed row is written to `fewshot_rows_to_remove_for_eval.csv`.
- The notebook also writes a fully reconciled `fewshot_pool_after_eval_exclusions.csv`.
- Model outputs, accuracy, explanations, and downstream experimental outcomes are never used to select evaluation items.

## Source benchmark

Official dataset: `MMMU/MMMU_Pro`  
Configuration: `standard (10 options)`  
Split: `test`

**Reference**

> Yue, X., Zheng, T., Ni, Y., Wang, Y., Zhang, K., Tong, S., Sun, Y., Yu, B., Zhang, G., Sun, H., Su, Y., Chen, W., & Neubig, G. (2024). *MMMU-Pro: A More Robust Multi-discipline Multimodal Understanding Benchmark*. arXiv:2409.02813.


## Manuscript-ready sampling description

The notebook generates the final wording dynamically in `METHODS.txt`. The protocol is:

> **Evaluation subset construction.** We constructed a fixed 300-item subset
> from the pinned MMMU-Pro Standard (10 options) test split using deterministic
> stratified random sampling without replacement. Exactly 10 items were selected
> from each of the benchmark's 30 subjects, with a nominal within-subject target
> of 3 Easy, 4 Medium, and 3 Hard items. After excluding the separately curated
> few-shot candidate pool, any shortage in Easy or Hard was first repaired by
> deterministically reclaiming the minimum required number of examples from the
> same subject and same difficulty in that few-shot pool. Reclaimed examples were
> permanently removed from the final few-shot pool. If a residual Easy or Hard
> shortage remained because the complete pinned benchmark itself did not contain
> enough examples, only that irreducible deficit was transferred to Medium while
> preserving exactly 10 items for the subject. Candidate IDs were canonically
> sorted before sampling, and NumPy PCG64 was initialized with fixed seeds. The
> final evaluation set was validated to contain 300 unique items, exactly 10 per
> subject, maximal preservation of the nominal 3/4/3 allocation under source
> availability, and zero overlap with the final few-shot pool.


In [1]:
# Configuration: constants defining the sampling protocol.
from pathlib import Path

MASTER_SEED = 42

# Independent deterministic streams.
SAMPLING_SEED = MASTER_SEED
RECLAIM_SEED = MASTER_SEED + 1
EVALUATION_ORDER_SEED = MASTER_SEED + 2

DATASET_ID = "MMMU/MMMU_Pro"
DATASET_CONFIG = "standard (10 options)"
DATASET_SPLIT = "test"
DATASET_REVISION = "563f3e8"

EXPECTED_SOURCE_ROWS = 1730
EXPECTED_SUBJECTS = 30
EXPECTED_PER_SUBJECT = 10
EXPECTED_SELECTED_ROWS = EXPECTED_SUBJECTS * EXPECTED_PER_SUBJECT

TARGET_QUOTA = {
    "Easy": 3,
    "Medium": 4,
    "Hard": 3,
}

EXCLUSION_FILENAME = "mmmu_pro_with_usable_explanations.csv"
EXCLUSION_EXPECTED_SHA256 = "1bf3cba4bf14355a5159e7ca100932e5cd00ae9227d359674f681156f8182b67"
EXCLUSION_EXPECTED_ROWS = 113
EXCLUSION_EXPECTED_UNIQUE_IDS = 113
STRICT_EXCLUSION_CHECKSUM = True

DIFFICULTY_ORDER = ["Easy", "Medium", "Hard"]

OUTPUT_DIR = Path("/kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Sampling protocol")
print("-----------------")
print("Master seed:", MASTER_SEED)
print("Sampling seed:", SAMPLING_SEED)
print("Reclaim seed:", RECLAIM_SEED)
print("Evaluation-order seed:", EVALUATION_ORDER_SEED)
print("Dataset:", DATASET_ID)
print("Config:", DATASET_CONFIG)
print("Split:", DATASET_SPLIT)
print("Pinned revision:", DATASET_REVISION)
print("Nominal quota:", TARGET_QUOTA)
print("Priority: reclaim missing Easy/Hard from few-shot first")
print("Fallback: irreducible residual deficit -> Medium")
print("Expected final size:", EXPECTED_SELECTED_ROWS)
print("Output directory:", OUTPUT_DIR)


Sampling protocol
-----------------
Master seed: 42
Sampling seed: 42
Reclaim seed: 43
Evaluation-order seed: 44
Dataset: MMMU/MMMU_Pro
Config: standard (10 options)
Split: test
Pinned revision: 563f3e8
Nominal quota: {'Easy': 3, 'Medium': 4, 'Hard': 3}
Priority: reclaim missing Easy/Hard from few-shot first
Fallback: irreducible residual deficit -> Medium
Expected final size: 300
Output directory: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first


In [2]:
# Environment setup and version capture.
# This notebook is CPU-only; no GPU accelerator is required.

import sys
import os
import json
import hashlib
import platform
import subprocess
import importlib.util
from datetime import datetime, timezone

required_packages = {
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
}

missing = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )
else:
    print("All required packages are already installed.")

import numpy as np
import pandas as pd
import datasets
import pyarrow
import huggingface_hub
from huggingface_hub import HfApi
from datasets import load_dataset

SOFTWARE_VERSIONS = {
    "python": sys.version.replace("\n", " "),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "datasets": datasets.__version__,
    "pyarrow": pyarrow.__version__,
    "huggingface_hub": huggingface_hub.__version__,
}

print(json.dumps(SOFTWARE_VERSIONS, indent=2))

All required packages are already installed.
{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "numpy": "2.0.2",
  "pandas": "2.3.3",
  "datasets": "5.0.0",
  "pyarrow": "24.0.0",
  "huggingface_hub": "1.11.0"
}


## 1. Locate and authenticate the few-shot exclusion file

The exclusion file is not treated as an informal auxiliary input. It is part of the study protocol.

The notebook therefore:

1. searches Kaggle Input for the exact expected filename;
2. fails if zero or multiple ambiguous copies are found;
3. computes a SHA-256 checksum;
4. verifies the expected row count and uniqueness of `id`;
5. later verifies that every excluded ID is a valid MMMU-Pro item.

For the exact exclusion file supplied for this study, the expected profile is:

- **113 rows**
- **113 unique MMMU-Pro IDs**
- difficulty counts: **41 Easy, 44 Medium, 28 Hard**
- represented subjects: **21**

The number of subjects represented in the exclusion set does **not** need to be 30; it is only a removal set. The final evaluation subset is separately required to cover all 30 benchmark subjects.

### Implementation robustness note

Kaggle input paths may be supplied either as filenames or as full absolute
paths. The exclusion-file resolver below deliberately supports both forms
and avoids passing an absolute path to `Path.rglob`, whose pattern argument
must be relative. Ambiguous duplicate copies still cause a hard failure so
that the provenance of the exclusion set remains uniquely defined.

In [3]:
# Locate the exact exclusion CSV in Kaggle Input.
#
# Robustness note:
# pathlib.Path.rglob() accepts only a RELATIVE pattern. To avoid accidental
# failures when EXCLUSION_FILENAME is changed to an absolute path (or when a
# stale notebook variable contains one), file discovery is implemented with
# an explicit path check plus deterministic os.walk() traversal.

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


INPUT_ROOT = Path("/kaggle/input").resolve()

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "/kaggle/input does not exist. This notebook is intended for Kaggle."
    )


def locate_exclusion_file(
    input_root: Path,
    file_spec: str | os.PathLike,
) -> Path:
    """
    Resolve exactly one exclusion CSV.

    Supported inputs
    ----------------
    1. A basename, e.g. "mmmu_pro_with_usable_explanations.csv".
       The Kaggle Input tree is searched recursively.
    2. A relative path under /kaggle/input.
    3. An explicit absolute path.

    The function intentionally fails on ambiguity rather than choosing an
    arbitrary copy, preserving experimental provenance.
    """
    input_root = Path(input_root).resolve()
    spec = Path(str(file_spec))

    # Case 1: explicit absolute path.
    if spec.is_absolute():
        resolved = spec.resolve()
        if not resolved.is_file():
            raise FileNotFoundError(
                f"Explicit exclusion path does not exist: {resolved}"
            )
        return resolved

    # Case 2: an exact relative path under Kaggle Input.
    exact_relative = (input_root / spec).resolve()
    if exact_relative.is_file():
        return exact_relative

    # Case 3: recursively find the basename.
    target_name = spec.name
    candidates = []

    for root, dirnames, filenames in os.walk(input_root):
        # Sorting makes discovery deterministic across filesystems.
        dirnames.sort()
        filenames.sort()

        if target_name in filenames:
            candidates.append(
                (Path(root) / target_name).resolve()
            )

    # Deduplicate while preserving deterministic lexical order.
    candidates = sorted(set(candidates), key=lambda p: str(p))

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find {target_name!r} under {input_root}. "
            "Upload/add the exclusion CSV as a Kaggle Input before running."
        )

    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple copies of the exclusion CSV were found. "
            "To avoid ambiguous provenance, keep exactly one copy attached.\n"
            + "\n".join(str(p) for p in candidates)
        )

    return candidates[0]


print("Configured exclusion file spec:", repr(EXCLUSION_FILENAME))
print("Kaggle Input root:", INPUT_ROOT)

EXCLUSION_PATH = locate_exclusion_file(
    INPUT_ROOT,
    EXCLUSION_FILENAME,
)

EXCLUSION_SHA256 = sha256_file(EXCLUSION_PATH)

print("Resolved exclusion file:", EXCLUSION_PATH)
print("SHA-256:", EXCLUSION_SHA256)

if STRICT_EXCLUSION_CHECKSUM:
    assert EXCLUSION_SHA256 == EXCLUSION_EXPECTED_SHA256, (
        "The exclusion CSV checksum does not match the file used to define "
        "this experimental protocol.\n"
        f"Expected: {EXCLUSION_EXPECTED_SHA256}\n"
        f"Observed: {EXCLUSION_SHA256}"
    )

exclude_df = pd.read_csv(EXCLUSION_PATH)

required_exclusion_columns = {
    "id",
    "subject",
    "difficulty",
    "question",
}

missing_columns = sorted(
    required_exclusion_columns - set(exclude_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Exclusion CSV is missing required columns: {missing_columns}"
    )

if len(exclude_df) != EXCLUSION_EXPECTED_ROWS:
    raise AssertionError(
        f"Expected {EXCLUSION_EXPECTED_ROWS} exclusion rows, "
        f"observed {len(exclude_df)}."
    )

if exclude_df["id"].isna().any():
    raise AssertionError("Exclusion CSV contains missing IDs.")

if exclude_df["id"].duplicated().any():
    duplicate_ids = exclude_df.loc[
        exclude_df["id"].duplicated(keep=False), "id"
    ].tolist()
    raise AssertionError(
        f"Exclusion CSV contains duplicate IDs: {duplicate_ids[:20]}"
    )

if exclude_df["id"].nunique() != EXCLUSION_EXPECTED_UNIQUE_IDS:
    raise AssertionError("Unexpected number of unique exclusion IDs.")

print("\nExclusion-file profile")
print("----------------------")
print("Rows:", len(exclude_df))
print("Unique IDs:", exclude_df["id"].nunique())
print("Subjects represented:", exclude_df["subject"].nunique())
print("\nDifficulty counts:")
print(exclude_df["difficulty"].value_counts(dropna=False).sort_index())

Configured exclusion file spec: 'mmmu_pro_with_usable_explanations.csv'
Kaggle Input root: /kaggle/input
Resolved exclusion file: /kaggle/input/datasets/vernvern/sanasdataset/mmmu_pro_with_usable_explanations.csv
SHA-256: 1bf3cba4bf14355a5159e7ca100932e5cd00ae9227d359674f681156f8182b67

Exclusion-file profile
----------------------
Rows: 113
Unique IDs: 113
Subjects represented: 21

Difficulty counts:
difficulty
Easy      41
Hard      28
Medium    44
Name: count, dtype: int64


## 2. Resolve and load the pinned MMMU-Pro source revision

For reproducibility, the notebook does **not** load an unversioned moving `main` branch. The configured revision is first resolved through the Hugging Face Hub to its full commit SHA; that immutable SHA is then passed to `load_dataset`.

The resolved repository SHA and the Hugging Face `Dataset` fingerprint are both persisted in the final audit file.

In [4]:
# Resolve the pinned revision to an immutable full SHA, then load the source data.

hf_token = os.environ.get("HF_TOKEN") or None
api = HfApi(token=hf_token)

dataset_info = api.dataset_info(
    repo_id=DATASET_ID,
    revision=DATASET_REVISION,
)

RESOLVED_DATASET_SHA = dataset_info.sha

if not RESOLVED_DATASET_SHA:
    raise RuntimeError("Could not resolve the MMMU-Pro revision to a full SHA.")

print("Configured revision:", DATASET_REVISION)
print("Resolved full SHA:", RESOLVED_DATASET_SHA)

ds = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
    revision=RESOLVED_DATASET_SHA,
)

SOURCE_FINGERPRINT = ds._fingerprint

print("\nDataset loaded:")
print(ds)
print("Rows:", len(ds))
print("Fingerprint:", SOURCE_FINGERPRINT)

if len(ds) != EXPECTED_SOURCE_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_SOURCE_ROWS} source rows, observed {len(ds)}."
    )

Configured revision: 563f3e8
Resolved full SHA: 563f3e84bb3b90893083a1f039cfa13077f2302b


README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]


Dataset loaded:
Dataset({
    features: ['id', 'question', 'options', 'explanation', 'image_1', 'image_2', 'image_3', 'image_4', 'image_5', 'image_6', 'image_7', 'img_type', 'answer', 'topic_difficulty', 'subject'],
    num_rows: 1730
})
Rows: 1730
Fingerprint: 344765d7d4eb9b07


## 3. Validate the source schema and stratification metadata

The sampling design depends only on:

- `id`
- `subject`
- `topic_difficulty`

Before any item is sampled, this notebook verifies that:

- `id` is complete and globally unique;
- exactly 30 subjects are present;
- `topic_difficulty` contains only the expected levels `Easy`, `Medium`, and `Hard`;
- every subject has records available for all three difficulty levels.

This prevents silent changes in the source schema or category labels from changing the experimental subset.

In [5]:
# Construct a lightweight metadata table without materializing image bytes in pandas.

required_source_columns = {
    "id",
    "question",
    "options",
    "explanation",
    "answer",
    "topic_difficulty",
    "subject",
    "img_type",
}

missing_source_columns = sorted(
    required_source_columns - set(ds.column_names)
)

if missing_source_columns:
    raise RuntimeError(
        f"MMMU-Pro source is missing columns: {missing_source_columns}"
    )

meta = pd.DataFrame({
    "source_row_index": np.arange(len(ds), dtype=np.int64),
    "id": ds["id"],
    "subject": ds["subject"],
    "difficulty_raw": ds["topic_difficulty"],
})

if meta["id"].isna().any():
    raise AssertionError("Source dataset contains missing IDs.")

if meta["id"].duplicated().any():
    raise AssertionError("Source dataset IDs are not unique.")


def normalize_difficulty(value: object) -> str:
    key = str(value).strip().casefold()
    mapping = {
        "easy": "Easy",
        "medium": "Medium",
        "hard": "Hard",
    }
    if key not in mapping:
        raise ValueError(f"Unexpected difficulty label: {value!r}")
    return mapping[key]


meta["difficulty"] = meta["difficulty_raw"].map(normalize_difficulty)

subjects = sorted(meta["subject"].astype(str).unique().tolist())
difficulties = sorted(meta["difficulty"].unique().tolist())

if len(subjects) != EXPECTED_SUBJECTS:
    raise AssertionError(
        f"Expected {EXPECTED_SUBJECTS} subjects, observed {len(subjects)}."
    )

if set(difficulties) != set(DIFFICULTY_ORDER):
    raise AssertionError(
        f"Unexpected difficulty levels: {difficulties}"
    )

subject_difficulty_source = (
    meta.groupby(["subject", "difficulty"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=subjects, columns=DIFFICULTY_ORDER, fill_value=0)
)

if (subject_difficulty_source == 0).any().any():
    raise AssertionError(
        "At least one subject × difficulty stratum is empty in the source dataset."
    )

print("Subjects:", len(subjects))
print(subjects)
print("\nSource subject × difficulty counts:")
display(subject_difficulty_source)

Subjects: 30
['Accounting', 'Agriculture', 'Architecture_and_Engineering', 'Art', 'Art_Theory', 'Basic_Medical_Science', 'Biology', 'Chemistry', 'Clinical_Medicine', 'Computer_Science', 'Design', 'Diagnostics_and_Laboratory_Medicine', 'Economics', 'Electronics', 'Energy_and_Power', 'Finance', 'Geography', 'History', 'Literature', 'Manage', 'Marketing', 'Materials', 'Math', 'Mechanical_Engineering', 'Music', 'Pharmacy', 'Physics', 'Psychology', 'Public_Health', 'Sociology']

Source subject × difficulty counts:


difficulty,Easy,Medium,Hard
subject,,,
Accounting,22,28,8
Agriculture,17,7,36
Architecture_and_Engineering,7,23,30
Art,26,21,6
Art_Theory,29,20,6
Basic_Medical_Science,16,29,7
Biology,24,29,6
Chemistry,17,24,19
Clinical_Medicine,10,39,10


## 4. Verify the exclusion file against MMMU-Pro before removing items

Exclusion is performed by the benchmark's unique `id` field.

Before exclusion, the notebook verifies that every ID in the supplied few-shot file exists in the pinned MMMU-Pro source. When auxiliary provenance fields are available, it also checks consistency of:

- source row index;
- subject;
- difficulty;
- normalized question text.

This is a guard against accidentally attaching a few-shot file from a different benchmark configuration or dataset revision.

In [6]:
import re
import unicodedata


def normalize_text(value: object) -> str:
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


source_id_set = set(meta["id"].astype(str))
exclude_ids = set(exclude_df["id"].astype(str))

unknown_exclusion_ids = sorted(exclude_ids - source_id_set)

if unknown_exclusion_ids:
    raise AssertionError(
        "Some exclusion IDs do not exist in the pinned MMMU-Pro source. "
        f"First examples: {unknown_exclusion_ids[:20]}"
    )

# Map source metadata by ID.
source_lookup = meta.set_index("id", drop=False)

# Verify source row index if the column is available.
row_index_mismatches = []

if "source_row_index" in exclude_df.columns:
    for rec in exclude_df[["id", "source_row_index"]].itertuples(index=False):
        if pd.isna(rec.source_row_index):
            continue
        expected_idx = int(source_lookup.loc[str(rec.id), "source_row_index"])
        observed_idx = int(rec.source_row_index)
        if expected_idx != observed_idx:
            row_index_mismatches.append(
                (str(rec.id), observed_idx, expected_idx)
            )

if row_index_mismatches:
    raise AssertionError(
        "Exclusion-file source_row_index does not match the pinned dataset. "
        f"First mismatches: {row_index_mismatches[:10]}"
    )

# Subject consistency.
subject_mismatches = []

for rec in exclude_df[["id", "subject"]].itertuples(index=False):
    source_subject = str(source_lookup.loc[str(rec.id), "subject"])
    if str(rec.subject) != source_subject:
        subject_mismatches.append(
            (str(rec.id), str(rec.subject), source_subject)
        )

if subject_mismatches:
    raise AssertionError(
        f"Subject mismatches detected: {subject_mismatches[:10]}"
    )

# Difficulty consistency.
difficulty_mismatches = []

for rec in exclude_df[["id", "difficulty"]].itertuples(index=False):
    source_difficulty = str(source_lookup.loc[str(rec.id), "difficulty"])
    exclusion_difficulty = normalize_difficulty(rec.difficulty)
    if exclusion_difficulty != source_difficulty:
        difficulty_mismatches.append(
            (str(rec.id), exclusion_difficulty, source_difficulty)
        )

if difficulty_mismatches:
    raise AssertionError(
        f"Difficulty mismatches detected: {difficulty_mismatches[:10]}"
    )

# Question consistency.
question_by_id = dict(zip(ds["id"], ds["question"]))
question_mismatches = []

for rec in exclude_df[["id", "question"]].itertuples(index=False):
    source_q = normalize_text(question_by_id[str(rec.id)])
    exclusion_q = normalize_text(rec.question)
    if source_q != exclusion_q:
        question_mismatches.append(str(rec.id))

if question_mismatches:
    raise AssertionError(
        "Question-text mismatches detected between the exclusion CSV and "
        f"the pinned source dataset. First IDs: {question_mismatches[:10]}"
    )

print("✅ All exclusion IDs exist in the pinned MMMU-Pro source.")
print("✅ Source row indices are consistent.")
print("✅ Subjects are consistent.")
print("✅ Difficulty labels are consistent.")
print("✅ Normalized question texts are consistent.")
print("Exclusion IDs:", len(exclude_ids))

✅ All exclusion IDs exist in the pinned MMMU-Pro source.
✅ Source row indices are consistent.
✅ Subjects are consistent.
✅ Difficulty labels are consistent.
✅ Normalized question texts are consistent.
Exclusion IDs: 113


## 5. Reconcile the few-shot pool before applying Medium compensation

This section implements the study's explicit priority order.

For each subject:

### Step 1 — preserve Easy = 3 if the source permits it

If fewer than 3 Easy examples remain after excluding the 113-item few-shot pool,
the notebook reclaims exactly the missing Easy examples from that pool, restricted to:

- the same subject;
- `difficulty == Easy`.

If the complete source itself contains fewer than 3 Easy examples, reclamation can
recover at most the source maximum. Only the remaining **irreducible** Easy deficit
is transferred to Medium.

### Step 2 — preserve Hard = 3 if the source permits it

The same procedure is applied independently to Hard:

- reclaim missing Hard rows from the same subject in the few-shot pool;
- if the full source still cannot reach 3, transfer the residual deficit to Medium.

### Step 3 — derive Medium

After Easy and Hard have been maximally restored,

\[
q_M = 10 - q_E - q_H.
\]

Thus Medium absorbs only source-level residual shortages, not avoidable shortages
caused by the few-shot exclusion.

### Step 4 — reconcile a possible Medium shortage

If the final Medium quota is itself undersupplied after the Easy/Hard reclamations,
the notebook deterministically reclaims the minimum required Medium rows from the
same subject in the original few-shot pool.

Every reclaimed Easy, Hard, or Medium row is permanently removed from the final
few-shot pool and written to:

- `fewshot_rows_to_remove_for_eval.csv`
- `fewshot_reclamation_audit.csv`

The ready-to-use remaining few-shot pool is written to:

- `fewshot_pool_after_eval_exclusions.csv`


In [7]:
# -------------------------------------------------------------------------
# Stage A: profile source availability and initial availability after excluding
# the original 113-item few-shot candidate pool.
# -------------------------------------------------------------------------

meta["excluded_in_original_fewshot_pool"] = meta["id"].isin(exclude_ids)

source_availability = (
    meta.groupby(["subject", "difficulty"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            index=subjects,
            columns=DIFFICULTY_ORDER,
            fill_value=0,
        )
)

nonexcluded_meta_initial = meta.loc[
    ~meta["excluded_in_original_fewshot_pool"]
].copy()

availability_after_original_exclusion = (
    nonexcluded_meta_initial.groupby(["subject", "difficulty"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            index=subjects,
            columns=DIFFICULTY_ORDER,
            fill_value=0,
        )
)

# -------------------------------------------------------------------------
# Stage B: reclaim Easy and Hard FIRST, deterministically and minimally.
# -------------------------------------------------------------------------

reclaim_rng = np.random.Generator(
    np.random.PCG64(RECLAIM_SEED)
)

reclaimed_ids = set()
reclaim_audit_rows = []


def reclaim_same_stratum(
    subject: str,
    difficulty: str,
    n_needed: int,
    reason: str,
) -> list[str]:
    """
    Reclaim up to n_needed rows from the original few-shot pool for exactly
    one subject × difficulty stratum. Selection is deterministic because
    candidates are canonically sorted before PCG64 sampling.
    """
    if n_needed <= 0:
        return []

    candidates = (
        meta.loc[
            meta["excluded_in_original_fewshot_pool"]
            & (~meta["id"].astype(str).isin(reclaimed_ids))
            & (meta["subject"] == subject)
            & (meta["difficulty"] == difficulty)
        ]
        .sort_values("id", kind="mergesort")
        .reset_index(drop=True)
    )

    n_take = min(int(n_needed), len(candidates))

    if n_take == 0:
        return []

    chosen_positions = reclaim_rng.choice(
        len(candidates),
        size=n_take,
        replace=False,
    )

    chosen = (
        candidates.iloc[chosen_positions]
        .sort_values("id", kind="mergesort")
        .reset_index(drop=True)
    )

    selected_ids_local = []

    for _, row in chosen.iterrows():
        item_id = str(row["id"])

        if item_id in reclaimed_ids:
            raise AssertionError(
                f"Duplicate reclaimed ID: {item_id}"
            )

        reclaimed_ids.add(item_id)
        selected_ids_local.append(item_id)

        reclaim_audit_rows.append({
            "id": item_id,
            "source_row_index": int(row["source_row_index"]),
            "subject": str(row["subject"]),
            "difficulty": str(row["difficulty"]),
            "reason": reason,
            "master_seed": MASTER_SEED,
            "reclaim_seed": RECLAIM_SEED,
        })

    return selected_ids_local


# Keep an audit of intended and actual restoration.
restoration_rows = []

for subject in subjects:
    for difficulty in ["Easy", "Hard"]:
        target = int(TARGET_QUOTA[difficulty])

        available_after_exclusion = int(
            availability_after_original_exclusion.loc[
                subject,
                difficulty,
            ]
        )

        source_total = int(
            source_availability.loc[
                subject,
                difficulty,
            ]
        )

        initial_deficit = max(
            0,
            target - available_after_exclusion,
        )

        # We can reclaim at most enough to reach the source-level maximum.
        maximum_repairable = max(
            0,
            min(target, source_total) - available_after_exclusion,
        )

        reclaimed_here = reclaim_same_stratum(
            subject=subject,
            difficulty=difficulty,
            n_needed=maximum_repairable,
            reason=(
                f"restore_{difficulty.lower()}_quota_before_medium_compensation"
            ),
        )

        available_after_reclaim = (
            available_after_exclusion
            + len(reclaimed_here)
        )

        effective_edge_quota = min(
            target,
            available_after_reclaim,
        )

        residual_deficit_to_medium = (
            target - effective_edge_quota
        )

        restoration_rows.append({
            "subject": subject,
            "difficulty": difficulty,
            "target": target,
            "source_total": source_total,
            "available_after_original_exclusion": available_after_exclusion,
            "initial_deficit": initial_deficit,
            "reclaimed_from_fewshot": len(reclaimed_here),
            "available_after_same_difficulty_reclaim": available_after_reclaim,
            "effective_quota": effective_edge_quota,
            "residual_deficit_transferred_to_medium": residual_deficit_to_medium,
        })

restoration_df = pd.DataFrame(restoration_rows)

print("Easy/Hard restoration audit:")
display(restoration_df)

# -------------------------------------------------------------------------
# Stage C: derive subject quotas AFTER Easy/Hard reclamation.
# -------------------------------------------------------------------------

edge_lookup = {
    (str(r.subject), str(r.difficulty)): int(r.effective_quota)
    for r in restoration_df.itertuples(index=False)
}

subject_quota_rows = []

for subject in subjects:
    effective_easy = edge_lookup[(subject, "Easy")]
    effective_hard = edge_lookup[(subject, "Hard")]

    residual_easy_deficit = (
        TARGET_QUOTA["Easy"] - effective_easy
    )
    residual_hard_deficit = (
        TARGET_QUOTA["Hard"] - effective_hard
    )

    effective_medium = (
        TARGET_QUOTA["Medium"]
        + residual_easy_deficit
        + residual_hard_deficit
    )

    assert (
        effective_easy
        + effective_medium
        + effective_hard
        == EXPECTED_PER_SUBJECT
    )

    subject_quota_rows.append({
        "subject": subject,
        "effective_easy": effective_easy,
        "effective_medium": effective_medium,
        "effective_hard": effective_hard,
        "residual_easy_deficit_to_medium": residual_easy_deficit,
        "residual_hard_deficit_to_medium": residual_hard_deficit,
    })

subject_quota_df = pd.DataFrame(
    subject_quota_rows
)

print("\nEffective quotas after Easy/Hard-first restoration:")
display(subject_quota_df)

# -------------------------------------------------------------------------
# Stage D: reconcile Medium if the derived Medium quota is undersupplied.
# -------------------------------------------------------------------------

def current_final_exclusion_ids():
    return set(exclude_ids) - set(reclaimed_ids)


medium_reconciliation_rows = []

for rec in subject_quota_df.sort_values(
    "subject",
    kind="mergesort",
).itertuples(index=False):

    subject = str(rec.subject)
    required_medium = int(rec.effective_medium)

    current_excluded = current_final_exclusion_ids()

    available_medium = int(
        meta.loc[
            (~meta["id"].isin(current_excluded))
            & (meta["subject"] == subject)
            & (meta["difficulty"] == "Medium")
        ].shape[0]
    )

    medium_deficit = max(
        0,
        required_medium - available_medium,
    )

    reclaimed_medium = reclaim_same_stratum(
        subject=subject,
        difficulty="Medium",
        n_needed=medium_deficit,
        reason="minimum_medium_reclaim_after_easy_hard_restoration",
    )

    available_medium_after = (
        available_medium + len(reclaimed_medium)
    )

    if available_medium_after < required_medium:
        raise RuntimeError(
            "Final Medium quota is infeasible even after reclaiming all "
            "available same-subject Medium rows from the few-shot pool. "
            f"subject={subject!r}, required={required_medium}, "
            f"available_after_reclaim={available_medium_after}."
        )

    medium_reconciliation_rows.append({
        "subject": subject,
        "required_medium": required_medium,
        "available_medium_before_reclaim": available_medium,
        "medium_deficit": medium_deficit,
        "medium_reclaimed_from_fewshot": len(reclaimed_medium),
        "available_medium_after_reclaim": available_medium_after,
    })

medium_reconciliation_df = pd.DataFrame(
    medium_reconciliation_rows
)

print("\nMedium reconciliation audit:")
display(medium_reconciliation_df)

# -------------------------------------------------------------------------
# Stage E: define FINAL few-shot and evaluation candidate pools.
# -------------------------------------------------------------------------

final_fewshot_exclude_ids = (
    set(exclude_ids) - set(reclaimed_ids)
)

eligible_meta = meta.loc[
    ~meta["id"].isin(final_fewshot_exclude_ids)
].copy()

final_availability = (
    eligible_meta.groupby(["subject", "difficulty"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            index=subjects,
            columns=DIFFICULTY_ORDER,
            fill_value=0,
        )
)

# Hard feasibility validation.
for rec in subject_quota_df.itertuples(index=False):
    subject = str(rec.subject)

    expected = {
        "Easy": int(rec.effective_easy),
        "Medium": int(rec.effective_medium),
        "Hard": int(rec.effective_hard),
    }

    for difficulty, required in expected.items():
        available = int(
            final_availability.loc[
                subject,
                difficulty,
            ]
        )

        if available < required:
            raise RuntimeError(
                "Reconciled candidate pool is still infeasible. "
                f"subject={subject!r}, difficulty={difficulty!r}, "
                f"available={available}, required={required}"
            )

# -------------------------------------------------------------------------
# Stage F: write explicit few-shot removal artifacts.
# -------------------------------------------------------------------------

reclaim_audit_df = pd.DataFrame(
    reclaim_audit_rows,
    columns=[
        "id",
        "source_row_index",
        "subject",
        "difficulty",
        "reason",
        "master_seed",
        "reclaim_seed",
    ],
)

fewshot_rows_to_remove = exclude_df.loc[
    exclude_df["id"].astype(str).isin(reclaimed_ids)
].copy()

fewshot_rows_to_remove["removal_reason"] = (
    "reclaimed_for_final_evaluation"
)
fewshot_rows_to_remove["master_seed"] = MASTER_SEED
fewshot_rows_to_remove["reclaim_seed"] = RECLAIM_SEED

fewshot_pool_after_eval_exclusions = exclude_df.loc[
    ~exclude_df["id"].astype(str).isin(reclaimed_ids)
].copy()

ROWS_TO_REMOVE_CSV = (
    OUTPUT_DIR / "fewshot_rows_to_remove_for_eval.csv"
)

UPDATED_FEWSHOT_CSV = (
    OUTPUT_DIR / "fewshot_pool_after_eval_exclusions.csv"
)

RECLAIM_AUDIT_CSV = (
    OUTPUT_DIR / "fewshot_reclamation_audit.csv"
)

RESTORATION_AUDIT_CSV = (
    OUTPUT_DIR / "easy_hard_restoration_audit.csv"
)

MEDIUM_AUDIT_CSV = (
    OUTPUT_DIR / "medium_reconciliation_audit.csv"
)

QUOTA_TABLE_CSV = (
    OUTPUT_DIR / "effective_quota_by_subject.csv"
)

FINAL_AVAILABILITY_CSV = (
    OUTPUT_DIR / "availability_after_final_reconciliation.csv"
)

fewshot_rows_to_remove.to_csv(
    ROWS_TO_REMOVE_CSV,
    index=False,
)

fewshot_pool_after_eval_exclusions.to_csv(
    UPDATED_FEWSHOT_CSV,
    index=False,
)

reclaim_audit_df.to_csv(
    RECLAIM_AUDIT_CSV,
    index=False,
)

restoration_df.to_csv(
    RESTORATION_AUDIT_CSV,
    index=False,
)

medium_reconciliation_df.to_csv(
    MEDIUM_AUDIT_CSV,
    index=False,
)

subject_quota_df.to_csv(
    QUOTA_TABLE_CSV,
    index=False,
)

final_availability.reset_index().to_csv(
    FINAL_AVAILABILITY_CSV,
    index=False,
)

assert set(
    fewshot_rows_to_remove["id"].astype(str)
) == set(reclaimed_ids)

assert set(
    fewshot_pool_after_eval_exclusions["id"].astype(str)
) == final_fewshot_exclude_ids

print("\nFew-shot reconciliation summary")
print("-------------------------------")
print("Original few-shot rows:", len(exclude_df))
print("Total rows reclaimed for evaluation:", len(reclaimed_ids))
print("Final few-shot rows:", len(fewshot_pool_after_eval_exclusions))
print("Removal-list file:", ROWS_TO_REMOVE_CSV)
print("Updated few-shot pool:", UPDATED_FEWSHOT_CSV)

if len(fewshot_rows_to_remove):
    print("\nREMOVE THESE ROWS FROM THE ORIGINAL 113-ITEM FEW-SHOT POOL:")
    display(
        fewshot_rows_to_remove[
            [
                c for c in [
                    "id",
                    "subject",
                    "difficulty",
                    "source_row_index",
                    "question",
                ]
                if c in fewshot_rows_to_remove.columns
            ]
        ]
    )
else:
    print("\nNo rows need to be removed from the original few-shot pool.")


Easy/Hard restoration audit:


,subject,difficulty,target,source_total,available_after_original_exclusion,initial_deficit,reclaimed_from_fewshot,available_after_same_difficulty_reclaim,effective_quota,residual_deficit_transferred_to_medium
0,Accounting,Easy,3,22,20,0,0,20,3,0
1,Accounting,Hard,3,8,7,0,0,7,3,0
2,Agriculture,Easy,3,17,11,0,0,11,3,0
3,Agriculture,Hard,3,36,26,0,0,26,3,0
4,Architecture_and_Engineering,Easy,3,7,7,0,0,7,3,0
5,Architecture_and_Engineering,Hard,3,30,30,0,0,30,3,0
6,Art,Easy,3,26,19,0,0,19,3,0
7,Art,Hard,3,6,6,0,0,6,3,0
8,Art_Theory,Easy,3,29,25,0,0,25,3,0
9,Art_Theory,Hard,3,6,6,0,0,6,3,0



Effective quotas after Easy/Hard-first restoration:


,subject,effective_easy,effective_medium,effective_hard,residual_easy_deficit_to_medium,residual_hard_deficit_to_medium
0,Accounting,3,4,3,0,0
1,Agriculture,3,4,3,0,0
2,Architecture_and_Engineering,3,4,3,0,0
3,Art,3,4,3,0,0
4,Art_Theory,3,4,3,0,0
5,Basic_Medical_Science,3,4,3,0,0
6,Biology,3,4,3,0,0
7,Chemistry,3,4,3,0,0
8,Clinical_Medicine,3,4,3,0,0
9,Computer_Science,3,4,3,0,0



Medium reconciliation audit:


,subject,required_medium,available_medium_before_reclaim,medium_deficit,medium_reclaimed_from_fewshot,available_medium_after_reclaim
0,Accounting,4,26,0,0,26
1,Agriculture,4,6,0,0,6
2,Architecture_and_Engineering,4,23,0,0,23
3,Art,4,17,0,0,17
4,Art_Theory,4,17,0,0,17
5,Basic_Medical_Science,4,28,0,0,28
6,Biology,4,29,0,0,29
7,Chemistry,4,24,0,0,24
8,Clinical_Medicine,4,39,0,0,39
9,Computer_Science,4,25,0,0,25



Few-shot reconciliation summary
-------------------------------
Original few-shot rows: 113
Total rows reclaimed for evaluation: 0
Final few-shot rows: 113
Removal-list file: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first/fewshot_rows_to_remove_for_eval.csv
Updated few-shot pool: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first/fewshot_pool_after_eval_exclusions.csv

No rows need to be removed from the original few-shot pool.


## 6. Deterministic stratified sampling after reconciliation

Sampling occurs only after:

1. Easy deficits caused by exclusion have been repaired from Easy few-shot rows;
2. Hard deficits caused by exclusion have been repaired from Hard few-shot rows;
3. only irreducible source-level Easy/Hard deficits have been transferred to Medium;
4. any remaining Medium shortage has been minimally reconciled.

Within each `subject × difficulty` stratum:

- candidate IDs are sorted lexicographically;
- sampling is without replacement;
- NumPy `Generator(PCG64)` uses `SAMPLING_SEED = 42`.

A separate deterministic seed (`EVALUATION_ORDER_SEED = 44`) is used only to
randomize the order of the already-selected 300 evaluation items.


In [8]:
# Canonical deterministic stratified sampling.

sampling_rng = np.random.Generator(
    np.random.PCG64(SAMPLING_SEED)
)

effective_quota_lookup = {}

for rec in subject_quota_df.itertuples(index=False):
    effective_quota_lookup[(str(rec.subject), "Easy")] = int(rec.effective_easy)
    effective_quota_lookup[(str(rec.subject), "Medium")] = int(rec.effective_medium)
    effective_quota_lookup[(str(rec.subject), "Hard")] = int(rec.effective_hard)

selected_parts = []

for subject in subjects:
    for difficulty in DIFFICULTY_ORDER:
        n_required = effective_quota_lookup[
            (subject, difficulty)
        ]

        stratum = (
            eligible_meta.loc[
                (eligible_meta["subject"] == subject)
                & (eligible_meta["difficulty"] == difficulty)
            ]
            .sort_values("id", kind="mergesort")
            .reset_index(drop=True)
        )

        if len(stratum) < n_required:
            raise RuntimeError(
                f"Infeasible final stratum: subject={subject!r}, "
                f"difficulty={difficulty!r}, "
                f"available={len(stratum)}, required={n_required}"
            )

        # IMPORTANT:
        # Reclaimed rows are not merely candidates. They were reclaimed
        # specifically to repair a deficit and must therefore be selected.
        mandatory = stratum.loc[
            stratum["id"].astype(str).isin(reclaimed_ids)
        ].copy()

        if len(mandatory) > n_required:
            raise RuntimeError(
                f"More mandatory reclaimed rows than quota in "
                f"{subject=} {difficulty=}: "
                f"mandatory={len(mandatory)}, quota={n_required}"
            )

        remaining_needed = n_required - len(mandatory)

        optional = stratum.loc[
            ~stratum["id"].astype(str).isin(reclaimed_ids)
        ].copy()

        if len(optional) < remaining_needed:
            raise RuntimeError(
                f"Insufficient optional rows after mandatory reclaim inclusion "
                f"for {subject=} {difficulty=}."
            )

        if remaining_needed > 0:
            chosen_positions = sampling_rng.choice(
                len(optional),
                size=remaining_needed,
                replace=False,
            )
            sampled_optional = optional.iloc[
                chosen_positions
            ].copy()
            chosen = pd.concat(
                [mandatory, sampled_optional],
                ignore_index=True,
            )
        else:
            chosen = mandatory.copy()

        chosen["master_seed"] = MASTER_SEED
        chosen["selection_seed"] = SAMPLING_SEED
        chosen["effective_quota_for_stratum"] = n_required
        chosen["was_reclaimed_from_fewshot"] = (
            chosen["id"].astype(str).isin(reclaimed_ids)
        )

        chosen = (
            chosen.sort_values("id", kind="mergesort")
                  .reset_index(drop=True)
        )

        chosen["selection_rank_within_stratum"] = np.arange(
            1,
            len(chosen) + 1,
            dtype=np.int64,
        )

        assert len(chosen) == n_required

        selected_parts.append(chosen)

selected = pd.concat(
    selected_parts,
    ignore_index=True,
)

if len(selected) != EXPECTED_SELECTED_ROWS:
    raise AssertionError(
        f"Expected {EXPECTED_SELECTED_ROWS} selected rows, got {len(selected)}."
    )

order_rng = np.random.Generator(
    np.random.PCG64(
        EVALUATION_ORDER_SEED
    )
)

permutation = order_rng.permutation(
    len(selected)
)

selected = (
    selected.iloc[permutation]
            .reset_index(drop=True)
)

selected["evaluation_order"] = np.arange(
    1,
    len(selected) + 1,
    dtype=np.int64,
)

print("Selected rows:", len(selected))
print(
    "Selected reclaimed rows:",
    int(selected["was_reclaimed_from_fewshot"].sum()),
)
display(selected.head(20))


Selected rows: 300
Selected reclaimed rows: 0


,source_row_index,id,subject,difficulty_raw,difficulty,excluded_in_original_fewshot_pool,master_seed,selection_seed,effective_quota_for_stratum,was_reclaimed_from_fewshot,selection_rank_within_stratum,evaluation_order
0,449,test_Public_Health_189,Public_Health,Hard,Hard,False,42,42,3,False,1,1
1,125,validation_Manage_30,Manage,Medium,Medium,False,42,42,4,False,4,2
2,177,test_Psychology_96,Psychology,Medium,Medium,False,42,42,4,False,4,3
3,305,test_Math_164,Math,Hard,Hard,False,42,42,3,False,1,4
4,640,validation_Finance_13,Finance,Easy,Easy,False,42,42,3,False,3,5
5,991,test_Public_Health_243,Public_Health,Medium,Medium,False,42,42,4,False,3,6
6,1367,test_Electronics_245,Electronics,Hard,Hard,False,42,42,3,False,2,7
7,1726,test_Geography_222,Geography,Easy,Easy,False,42,42,3,False,3,8
8,1339,validation_Geography_30,Geography,Medium,Medium,False,42,42,4,False,4,9
9,537,test_History_27,History,Hard,Hard,False,42,42,3,False,3,10


## 7. Hard validation of the final split

The final artifact is written only if all required invariants hold:

- exactly 300 evaluation rows;
- exactly 300 unique IDs;
- exactly 30 subjects;
- exactly 10 items per subject;
- Easy and Hard are restored from same-difficulty few-shot rows whenever possible;
- only residual source-level Easy/Hard deficits are transferred to Medium;
- every reclaimed row is actually present in evaluation;
- every reclaimed row is absent from the final few-shot pool;
- evaluation has zero overlap with the final few-shot pool;
- any overlap with the original 113-item pool is exactly the explicit reclaim/removal set.


In [9]:
# Final invariants.

assert len(selected) == EXPECTED_SELECTED_ROWS
assert selected["id"].nunique() == EXPECTED_SELECTED_ROWS
assert selected["subject"].nunique() == EXPECTED_SUBJECTS

subject_counts = (
    selected.groupby("subject")
        .size()
        .rename("n")
        .sort_index()
)

assert (
    subject_counts
    == EXPECTED_PER_SUBJECT
).all()

selected_pivot = (
    selected.groupby(
        ["subject", "difficulty"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=subjects,
        columns=DIFFICULTY_ORDER,
        fill_value=0,
    )
)

for rec in subject_quota_df.itertuples(index=False):
    subject = str(rec.subject)

    expected = {
        "Easy": int(rec.effective_easy),
        "Medium": int(rec.effective_medium),
        "Hard": int(rec.effective_hard),
    }

    for difficulty, required in expected.items():
        observed = int(
            selected_pivot.loc[
                subject,
                difficulty,
            ]
        )

        assert observed == required, (
            f"Quota mismatch: subject={subject!r}, difficulty={difficulty!r}, "
            f"observed={observed}, expected={required}"
        )

    assert expected["Medium"] == (
        EXPECTED_PER_SUBJECT
        - expected["Easy"]
        - expected["Hard"]
    )

# Verify the priority rule for Easy and Hard:
# effective quota must equal min(target, full source count), because we reclaimed
# excluded same-difficulty rows before compensating with Medium.
for subject in subjects:
    expected_easy = min(
        TARGET_QUOTA["Easy"],
        int(source_availability.loc[subject, "Easy"]),
    )

    expected_hard = min(
        TARGET_QUOTA["Hard"],
        int(source_availability.loc[subject, "Hard"]),
    )

    assert int(
        selected_pivot.loc[subject, "Easy"]
    ) == expected_easy

    assert int(
        selected_pivot.loc[subject, "Hard"]
    ) == expected_hard

selected_ids = set(
    selected["id"].astype(str)
)

# Every reclaimed row MUST be selected.
assert set(reclaimed_ids).issubset(
    selected_ids
)

# Zero overlap with the FINAL few-shot pool.
assert selected_ids.isdisjoint(
    final_fewshot_exclude_ids
)

# Original 113-pool overlap must equal reclaim set exactly.
selected_from_original_fewshot = (
    selected_ids & exclude_ids
)

assert (
    selected_from_original_fewshot
    == set(reclaimed_ids)
)

assert set(
    fewshot_rows_to_remove["id"].astype(str)
) == set(reclaimed_ids)

assert selected_ids.isdisjoint(
    set(
        fewshot_pool_after_eval_exclusions[
            "id"
        ].astype(str)
    )
)

assert selected["source_row_index"].nunique() == EXPECTED_SELECTED_ROWS

assert selected["source_row_index"].between(
    0,
    len(ds) - 1,
).all()

assert selected["evaluation_order"].tolist() == list(
    range(
        1,
        EXPECTED_SELECTED_ROWS + 1,
    )
)

print("✅ FINAL VALIDATION PASSED")
print(f"✅ Total rows: {len(selected)}")
print(f"✅ Unique IDs: {selected['id'].nunique()}")
print(f"✅ Subjects: {selected['subject'].nunique()}")
print("✅ Exactly 10 items per subject")
print("✅ Easy restored from Easy few-shot rows whenever source permits")
print("✅ Hard restored from Hard few-shot rows whenever source permits")
print("✅ Only irreducible source deficits transferred to Medium")
print(f"✅ Reclaimed rows: {len(reclaimed_ids)}")
print("✅ Every reclaimed row is selected into evaluation")
print("✅ Zero overlap with final few-shot pool")
print("✅ Original 113-pool overlap equals explicit removal list")
print("✅ Evaluation order is a complete 1..300 permutation")

print("\nFinal selected counts:")
display(selected_pivot)

print("\nEasy/Hard restoration audit:")
display(restoration_df)

print("\nEffective quota table:")
display(subject_quota_df)


✅ FINAL VALIDATION PASSED
✅ Total rows: 300
✅ Unique IDs: 300
✅ Subjects: 30
✅ Exactly 10 items per subject
✅ Easy restored from Easy few-shot rows whenever source permits
✅ Hard restored from Hard few-shot rows whenever source permits
✅ Only irreducible source deficits transferred to Medium
✅ Reclaimed rows: 0
✅ Every reclaimed row is selected into evaluation
✅ Zero overlap with final few-shot pool
✅ Original 113-pool overlap equals explicit removal list
✅ Evaluation order is a complete 1..300 permutation

Final selected counts:


difficulty,Easy,Medium,Hard
subject,,,
Accounting,3,4,3
Agriculture,3,4,3
Architecture_and_Engineering,3,4,3
Art,3,4,3
Art_Theory,3,4,3
Basic_Medical_Science,3,4,3
Biology,3,4,3
Chemistry,3,4,3
Clinical_Medicine,3,4,3



Easy/Hard restoration audit:


,subject,difficulty,target,source_total,available_after_original_exclusion,initial_deficit,reclaimed_from_fewshot,available_after_same_difficulty_reclaim,effective_quota,residual_deficit_transferred_to_medium
0,Accounting,Easy,3,22,20,0,0,20,3,0
1,Accounting,Hard,3,8,7,0,0,7,3,0
2,Agriculture,Easy,3,17,11,0,0,11,3,0
3,Agriculture,Hard,3,36,26,0,0,26,3,0
4,Architecture_and_Engineering,Easy,3,7,7,0,0,7,3,0
5,Architecture_and_Engineering,Hard,3,30,30,0,0,30,3,0
6,Art,Easy,3,26,19,0,0,19,3,0
7,Art,Hard,3,6,6,0,0,6,3,0
8,Art_Theory,Easy,3,29,25,0,0,25,3,0
9,Art_Theory,Hard,3,6,6,0,0,6,3,0



Effective quota table:


,subject,effective_easy,effective_medium,effective_hard,residual_easy_deficit_to_medium,residual_hard_deficit_to_medium
0,Accounting,3,4,3,0,0
1,Agriculture,3,4,3,0,0
2,Architecture_and_Engineering,3,4,3,0,0
3,Art,3,4,3,0,0
4,Art_Theory,3,4,3,0,0
5,Basic_Medical_Science,3,4,3,0,0
6,Biology,3,4,3,0,0
7,Chemistry,3,4,3,0,0
8,Clinical_Medicine,3,4,3,0,0
9,Computer_Science,3,4,3,0,0


## 8. Persist publication and audit artifacts

### Evaluation artifacts

- `mmmu_pro_eval300_manifest.csv`
- `mmmu_pro_eval300_records.jsonl`
- `mmmu_pro_eval300_hf/`
- `selected_ids.txt`
- `subject_difficulty_counts.csv`

### Reconciliation artifacts

- **`fewshot_rows_to_remove_for_eval.csv`** — exact original few-shot rows reclaimed into evaluation; these must be removed from the few-shot pool.
- **`fewshot_pool_after_eval_exclusions.csv`** — ready-to-use final few-shot candidate pool.
- `fewshot_reclamation_audit.csv`
- `easy_hard_restoration_audit.csv`
- `medium_reconciliation_audit.csv`
- `effective_quota_by_subject.csv`
- `availability_after_final_reconciliation.csv`

### Reproducibility artifacts

- `selection_audit.json`
- `METHODS.txt`
- `CITATION.bib`

The primary multimodal evaluation artifact removes MMMU-Pro's source
`explanation` field to reduce accidental leakage. Gold `answer` is retained only
for scoring and must not be inserted into the evaluation prompt.


In [10]:
# Build and save publication artifacts.

manifest_columns = [
    "evaluation_order",
    "source_row_index",
    "id",
    "subject",
    "difficulty",
    "master_seed",
    "selection_seed",
    "effective_quota_for_stratum",
    "was_reclaimed_from_fewshot",
    "selection_rank_within_stratum",
]

manifest = selected[
    manifest_columns
].copy()

MANIFEST_CSV = OUTPUT_DIR / "mmmu_pro_eval300_manifest.csv"
RECORDS_JSONL = OUTPUT_DIR / "mmmu_pro_eval300_records.jsonl"
HF_DATASET_DIR = OUTPUT_DIR / "mmmu_pro_eval300_hf"
SELECTED_IDS_TXT = OUTPUT_DIR / "selected_ids.txt"
COUNTS_CSV = OUTPUT_DIR / "subject_difficulty_counts.csv"
AUDIT_JSON = OUTPUT_DIR / "selection_audit.json"
METHODS_TXT = OUTPUT_DIR / "METHODS.txt"
CITATION_BIB = OUTPUT_DIR / "CITATION.bib"

manifest.to_csv(
    MANIFEST_CSV,
    index=False,
)

selected_pivot.reset_index().to_csv(
    COUNTS_CSV,
    index=False,
)

SELECTED_IDS_TXT.write_text(
    "\n".join(
        manifest["id"].astype(str).tolist()
    )
    + "\n",
    encoding="utf-8",
)

selected_indices = (
    manifest["source_row_index"]
        .astype(int)
        .tolist()
)

selected_ds = ds.select(
    selected_indices
)

selected_ds = selected_ds.add_column(
    "source_row_index",
    selected_indices,
)

selected_ds = selected_ds.add_column(
    "evaluation_order",
    manifest["evaluation_order"]
        .astype(int)
        .tolist(),
)

selected_ds = selected_ds.add_column(
    "difficulty",
    manifest["difficulty"].tolist(),
)

selected_ds = selected_ds.add_column(
    "selection_seed",
    [SAMPLING_SEED] * len(selected_ds),
)

selected_ds = selected_ds.add_column(
    "was_reclaimed_from_fewshot",
    manifest[
        "was_reclaimed_from_fewshot"
    ].astype(bool).tolist(),
)

if "explanation" in selected_ds.column_names:
    selected_ds = selected_ds.remove_columns(
        ["explanation"]
    )

assert selected_ds["id"] == manifest["id"].tolist()

selected_ds.save_to_disk(
    HF_DATASET_DIR
)

jsonl_fields = [
    "evaluation_order",
    "source_row_index",
    "id",
    "subject",
    "difficulty",
    "question",
    "options",
    "answer",
    "img_type",
    "was_reclaimed_from_fewshot",
]

with RECORDS_JSONL.open(
    "w",
    encoding="utf-8",
) as f:
    for i in range(len(selected_ds)):
        row = selected_ds[i]

        record = {
            field: row.get(field)
            for field in jsonl_fields
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

MANIFEST_SHA256 = sha256_file(
    MANIFEST_CSV
)

ROWS_TO_REMOVE_SHA256 = sha256_file(
    ROWS_TO_REMOVE_CSV
)

UPDATED_FEWSHOT_SHA256 = sha256_file(
    UPDATED_FEWSHOT_CSV
)

# Dynamically summarize source-level exceptions.
source_exception_lines = []

for rec in subject_quota_df.itertuples(index=False):
    if (
        int(rec.residual_easy_deficit_to_medium) > 0
        or int(rec.residual_hard_deficit_to_medium) > 0
    ):
        source_exception_lines.append(
            f"{rec.subject}: "
            f"{int(rec.effective_easy)} Easy, "
            f"{int(rec.effective_medium)} Medium, "
            f"{int(rec.effective_hard)} Hard"
        )

source_exception_text = (
    "; ".join(source_exception_lines)
    if source_exception_lines
    else "none"
)

methods_text = f"""Evaluation subset construction.
We constructed a fixed 300-item evaluation subset from the pinned MMMU-Pro
Standard (10 options) test split using deterministic stratified random
sampling without replacement. Exactly 10 items were selected from each of
the benchmark's 30 subjects, with a nominal target of 3 Easy, 4 Medium, and
3 Hard items per subject. After removing the separately curated few-shot
candidate pool, any Easy or Hard shortage was first repaired by reclaiming
the minimum required number of examples from the same subject and the same
difficulty in that pool. All reclaimed examples were permanently removed
from the final few-shot pool and recorded in
fewshot_rows_to_remove_for_eval.csv. Only when the complete pinned source
itself contained fewer than three examples at Easy or Hard was the remaining
irreducible deficit transferred to Medium. The resulting source-level
exceptions were: {source_exception_text}. If the resulting Medium quota was
undersupplied, only the minimum required same-subject Medium examples were
reclaimed from the few-shot pool. Candidate rows were canonically sorted by
benchmark ID; NumPy PCG64 used seed {SAMPLING_SEED} for final stratified
sampling and independent fixed seeds for reclamation and evaluation ordering.
The resulting evaluation set contained 300 unique items, exactly 10 per
subject, maximally preserved the nominal 3/4/3 allocation under the pinned
source availability, and had zero overlap with the final few-shot pool.
"""

METHODS_TXT.write_text(
    methods_text,
    encoding="utf-8",
)

citation_bib = r"""@article{yue2024mmmupro,
  title   = {MMMU-Pro: A More Robust Multi-discipline Multimodal Understanding Benchmark},
  author  = {Xiang Yue and Tianyu Zheng and Yuansheng Ni and Yubo Wang and Kai Zhang and Shengbang Tong and Yuxuan Sun and Botao Yu and Ge Zhang and Huan Sun and Yu Su and Wenhu Chen and Graham Neubig},
  journal = {arXiv preprint arXiv:2409.02813},
  year    = {2024}
}
"""

CITATION_BIB.write_text(
    citation_bib,
    encoding="utf-8",
)

audit = {
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "protocol_name": (
        "MMMU-Pro Eval300: reclaim Easy/Hard from few-shot first; "
        "Medium only for irreducible residual deficits"
    ),

    "sampling_method": (
        "deterministic stratified random sampling without replacement"
    ),

    "priority_rule": {
        "easy": (
            "restore to 3 from same-subject Easy few-shot rows when possible; "
            "residual source-level deficit -> Medium"
        ),
        "hard": (
            "restore to 3 from same-subject Hard few-shot rows when possible; "
            "residual source-level deficit -> Medium"
        ),
        "medium": (
            "4 + residual Easy deficit + residual Hard deficit; "
            "if undersupplied, minimally reclaim same-subject Medium"
        ),
    },

    "rng": "numpy.random.Generator(PCG64)",

    "seeds": {
        "master_seed": MASTER_SEED,
        "sampling_seed": SAMPLING_SEED,
        "reclaim_seed": RECLAIM_SEED,
        "evaluation_order_seed": EVALUATION_ORDER_SEED,
    },

    "source": {
        "dataset_id": DATASET_ID,
        "config": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "configured_revision": DATASET_REVISION,
        "resolved_revision_sha": RESOLVED_DATASET_SHA,
        "dataset_fingerprint": SOURCE_FINGERPRINT,
        "source_rows": len(ds),
        "source_subjects": len(subjects),
    },

    "fewshot_pool": {
        "original_filename": EXCLUSION_PATH.name,
        "original_sha256": EXCLUSION_SHA256,
        "original_rows": len(exclude_df),
        "reclaimed_rows": len(reclaimed_ids),
        "reclaimed_ids": sorted(reclaimed_ids),
        "rows_to_remove_filename": ROWS_TO_REMOVE_CSV.name,
        "rows_to_remove_sha256": ROWS_TO_REMOVE_SHA256,
        "updated_fewshot_filename": UPDATED_FEWSHOT_CSV.name,
        "updated_fewshot_sha256": UPDATED_FEWSHOT_SHA256,
        "updated_fewshot_rows": len(
            fewshot_pool_after_eval_exclusions
        ),
    },

    "effective_quota_by_subject": {
        str(rec.subject): {
            "Easy": int(rec.effective_easy),
            "Medium": int(rec.effective_medium),
            "Hard": int(rec.effective_hard),
            "residual_easy_deficit_to_medium": int(
                rec.residual_easy_deficit_to_medium
            ),
            "residual_hard_deficit_to_medium": int(
                rec.residual_hard_deficit_to_medium
            ),
        }
        for rec in subject_quota_df.itertuples(index=False)
    },

    "result": {
        "selected_rows": len(manifest),
        "unique_selected_ids": int(
            manifest["id"].nunique()
        ),
        "selected_subjects": int(
            manifest["subject"].nunique()
        ),
        "overlap_with_original_113_pool": len(
            set(manifest["id"].astype(str))
            & exclude_ids
        ),
        "overlap_with_final_fewshot_pool": len(
            set(manifest["id"].astype(str))
            & final_fewshot_exclude_ids
        ),
        "manifest_sha256": MANIFEST_SHA256,
    },

    "software_versions": SOFTWARE_VERSIONS,
}

AUDIT_JSON.write_text(
    json.dumps(
        audit,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Artifacts written to:", OUTPUT_DIR)
print("Manifest SHA-256:", MANIFEST_SHA256)
print("Few-shot removal file:", ROWS_TO_REMOVE_CSV)
print("Updated few-shot pool:", UPDATED_FEWSHOT_CSV)


Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Artifacts written to: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first
Manifest SHA-256: abc0513ab722ed0a476142abd36a4cebe69d72affc25cc918759ee1e9f37d1ac
Few-shot removal file: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first/fewshot_rows_to_remove_for_eval.csv
Updated few-shot pool: /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first/fewshot_pool_after_eval_exclusions.csv


## 9. Independent post-write publication audit

The final executable cell reloads the written artifacts and verifies:

- exact 300-item evaluation size;
- exact 30 × 10 subject balance;
- maximal restoration of Easy and Hard to 3 whenever the full pinned source permits;
- Medium compensation only for irreducible source-level residual deficits;
- inclusion of every reclaimed row in evaluation;
- zero overlap between evaluation and the final few-shot pool;
- exact equality between original-few-shot overlap and the explicit removal list;
- cryptographic integrity of the principal artifacts.


In [11]:
# Independent post-write verification.

saved_manifest = pd.read_csv(
    MANIFEST_CSV
)

saved_audit = json.loads(
    AUDIT_JSON.read_text(
        encoding="utf-8"
    )
)

saved_remove = pd.read_csv(
    ROWS_TO_REMOVE_CSV
)

saved_updated_fewshot = pd.read_csv(
    UPDATED_FEWSHOT_CSV
)

saved_quota = pd.read_csv(
    QUOTA_TABLE_CSV
)

assert sha256_file(MANIFEST_CSV) == (
    saved_audit["result"]["manifest_sha256"]
)

assert sha256_file(ROWS_TO_REMOVE_CSV) == (
    saved_audit["fewshot_pool"]["rows_to_remove_sha256"]
)

assert sha256_file(UPDATED_FEWSHOT_CSV) == (
    saved_audit["fewshot_pool"]["updated_fewshot_sha256"]
)

assert len(saved_manifest) == EXPECTED_SELECTED_ROWS
assert saved_manifest["id"].nunique() == EXPECTED_SELECTED_ROWS
assert saved_manifest["subject"].nunique() == EXPECTED_SUBJECTS

saved_subject_counts = (
    saved_manifest.groupby("subject")
        .size()
)

assert (
    saved_subject_counts
    == EXPECTED_PER_SUBJECT
).all()

saved_counts = (
    saved_manifest.groupby(
        ["subject", "difficulty"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=subjects,
        columns=DIFFICULTY_ORDER,
        fill_value=0,
    )
)

for rec in saved_quota.itertuples(index=False):
    subject = str(rec.subject)

    expected_easy = min(
        TARGET_QUOTA["Easy"],
        int(source_availability.loc[subject, "Easy"]),
    )

    expected_hard = min(
        TARGET_QUOTA["Hard"],
        int(source_availability.loc[subject, "Hard"]),
    )

    expected_medium = (
        EXPECTED_PER_SUBJECT
        - expected_easy
        - expected_hard
    )

    assert int(
        saved_counts.loc[subject, "Easy"]
    ) == expected_easy

    assert int(
        saved_counts.loc[subject, "Hard"]
    ) == expected_hard

    assert int(
        saved_counts.loc[subject, "Medium"]
    ) == expected_medium

saved_eval_ids = set(
    saved_manifest["id"].astype(str)
)

saved_final_fewshot_ids = set(
    saved_updated_fewshot["id"].astype(str)
)

saved_remove_ids = set(
    saved_remove["id"].astype(str)
)

assert saved_eval_ids.isdisjoint(
    saved_final_fewshot_ids
)

assert saved_remove_ids == set(reclaimed_ids)

assert saved_remove_ids == (
    saved_eval_ids & exclude_ids
)

assert saved_final_fewshot_ids == (
    exclude_ids - set(reclaimed_ids)
)

print("=" * 100)
print("PUBLICATION DATASET CONSTRUCTION: VERIFIED")
print("=" * 100)
print("MMMU-Pro resolved revision :", RESOLVED_DATASET_SHA)
print("Dataset fingerprint        :", SOURCE_FINGERPRINT)
print("Master seed                :", MASTER_SEED)
print("Sampling seed              :", SAMPLING_SEED)
print("Reclaim seed               :", RECLAIM_SEED)
print("Evaluation-order seed      :", EVALUATION_ORDER_SEED)
print("Selected rows              :", len(saved_manifest))
print("Subjects                   :", saved_manifest["subject"].nunique())
print("Rows removed from few-shot :", len(saved_remove))
print("Final few-shot rows        :", len(saved_updated_fewshot))
print("Final few-shot overlap     :", 0)
print("Manifest SHA-256           :", sha256_file(MANIFEST_CSV))
print("Removal-list SHA-256       :", sha256_file(ROWS_TO_REMOVE_CSV))
print("Updated few-shot SHA-256   :", sha256_file(UPDATED_FEWSHOT_CSV))
print("Output directory           :", OUTPUT_DIR)
print("=" * 100)

print("\nFinal subject × difficulty counts:")
display(saved_counts)

print("\nRows that MUST be removed from the original 113-item few-shot pool:")
if len(saved_remove):
    display(saved_remove)
else:
    print("None.")

print("\nFiles:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_dir():
        print(f"[DIR ] {p.name}")
    else:
        print(
            f"[FILE] {p.name:56s} "
            f"{p.stat().st_size / 1024:.1f} KiB"
        )


PUBLICATION DATASET CONSTRUCTION: VERIFIED
MMMU-Pro resolved revision : 563f3e84bb3b90893083a1f039cfa13077f2302b
Dataset fingerprint        : 344765d7d4eb9b07
Master seed                : 42
Sampling seed              : 42
Reclaim seed               : 43
Evaluation-order seed      : 44
Selected rows              : 300
Subjects                   : 30
Rows removed from few-shot : 0
Final few-shot rows        : 113
Final few-shot overlap     : 0
Manifest SHA-256           : abc0513ab722ed0a476142abd36a4cebe69d72affc25cc918759ee1e9f37d1ac
Removal-list SHA-256       : 25f7893c13cc8dc4135bdde98dfa1b43ec96bb437dc719f3c5af2268b520e5e7
Updated few-shot SHA-256   : 1bf3cba4bf14355a5159e7ca100932e5cd00ae9227d359674f681156f8182b67
Output directory           : /kaggle/working/mmmu_pro_eval300_seed42_reclaim_easy_hard_first

Final subject × difficulty counts:


difficulty,Easy,Medium,Hard
subject,,,
Accounting,3,4,3
Agriculture,3,4,3
Architecture_and_Engineering,3,4,3
Art,3,4,3
Art_Theory,3,4,3
Basic_Medical_Science,3,4,3
Biology,3,4,3
Chemistry,3,4,3
Clinical_Medicine,3,4,3



Rows that MUST be removed from the original 113-item few-shot pool:
None.

Files:
[FILE] CITATION.bib                                             0.4 KiB
[FILE] METHODS.txt                                              1.5 KiB
[FILE] availability_after_final_reconciliation.csv              0.6 KiB
[FILE] easy_hard_restoration_audit.csv                          2.3 KiB
[FILE] effective_quota_by_subject.csv                           0.8 KiB
[FILE] fewshot_pool_after_eval_exclusions.csv                   128.2 KiB
[FILE] fewshot_reclamation_audit.csv                            0.1 KiB
[FILE] fewshot_rows_to_remove_for_eval.csv                      0.3 KiB
[FILE] medium_reconciliation_audit.csv                          0.8 KiB
[DIR ] mmmu_pro_eval300_hf
[FILE] mmmu_pro_eval300_manifest.csv                            19.1 KiB
[FILE] mmmu_pro_eval300_records.jsonl                           214.8 KiB
[FILE] selected_ids.txt                                         6.6 KiB
[FILE] selection_audi

## Notes for downstream evaluation and few-shot construction

1. Use `mmmu_pro_eval300_hf/` as the canonical evaluation dataset.
2. Use `fewshot_pool_after_eval_exclusions.csv` as the canonical few-shot candidate pool after this construction step.
3. Never reuse any ID listed in `fewshot_rows_to_remove_for_eval.csv` as a few-shot demonstration.
4. The nominal target is 3 Easy / 4 Medium / 3 Hard per subject.
5. If exclusion causes an Easy/Hard shortage, same-difficulty few-shot rows are reclaimed **before** any Medium compensation.
6. Medium compensates only a residual deficit that remains because the complete pinned source itself has fewer than 3 Easy or Hard examples.
7. The source `explanation` field is intentionally removed from the saved evaluation artifact.
8. Gold `answer` is retained only for scoring and must never be inserted into an evaluation prompt.
9. If the source revision or original 113-item few-shot pool changes, rerun the full construction pipeline and report the new hashes.

### Reporting checklist

Report at minimum:

- MMMU-Pro configuration and full revision SHA;
- source dataset fingerprint;
- fixed seeds;
- nominal 3/4/3 target;
- same-difficulty few-shot reclamation priority;
- source-level exceptions requiring Medium compensation;
- original few-shot pool SHA-256;
- exact reclaimed/removal IDs;
- updated few-shot pool SHA-256;
- final evaluation manifest SHA-256;
- zero overlap between final evaluation and final few-shot pools.
